# 📋 Cheatsheet Regex — Python / PySpark**Notebook interactif** pour apprendre et tester les expressions régulières.> **Comment l'utiliser** :> 1. Lis chaque section> 2. **Exécute** les cellules de code (Shift+Enter)> 3. **Modifie** les exemples pour tester tes propres cas> 4. Reviens-y comme référence quand tu en as besoin---## 📚 Sommaire1. [Setup & helper de test](#1-setup)2. [Caractères de base](#2-caracteres-de-base)3. [Quantificateurs](#3-quantificateurs)4. [Ancres de position](#4-ancres)5. [Classes de caractères `[...]`](#5-classes)6. [Groupes et alternatives](#6-groupes)7. [Négation et lookahead](#7-negation)8. [Raw string `r'...'`](#8-raw-string)9. [Recettes prêtes à l'emploi](#9-recettes)10. [Spécial PySpark](#10-pyspark)11. [Pièges classiques](#11-pieges)12. [Workflow d'écriture d'une regex](#12-workflow)

## 1. Setup & helper de test <a id='1-setup'></a>On va définir une **petite fonction utilitaire** qui teste une regex contre une liste de valeurs et affiche le résultat de manière lisible. Tu pourras l'utiliser tout au long du notebook.

In [ ]:
import redef test_regex(pattern, valeurs, description=""):    """Teste un pattern regex contre une liste de valeurs et affiche match/no-match."""    if description:        print(f"📌 {description}")    print(f"   Pattern : {pattern!r}")    print("-" * 50)    for v in valeurs:        match = re.fullmatch(pattern, v) if pattern.startswith('^') and pattern.endswith('$') else re.search(pattern, v)        symbole = "✅" if match else "❌"        print(f"   {symbole}  {v!r}")    print()

In [ ]:
# Test rapide pour vérifier que tout marchetest_regex(r'^\d{4}$', ['2022', '202', '20222', 'abc'], "Année (4 chiffres)")

## 2. Caractères de base <a id='2-caracteres-de-base'></a>| Symbole | Signifie ||---|---|| `.` | n'importe quel caractère (sauf `\n`) || `\d` | un chiffre (0-9) || `\D` | tout SAUF un chiffre || `\w` | lettre / chiffre / `_` || `\W` | tout SAUF alphanumérique || `\s` | espace, tab, retour ligne || `\S` | tout SAUF un espace |🧠 Les versions **majuscules** sont l'**inverse** des minuscules.

In [ ]:
# Le point . = n'importe quel caractèretest_regex(r'^a.c$', ['abc', 'a1c', 'a-c', 'ac', 'abbc'], "a.c")

In [ ]:
# \d = un chiffretest_regex(r'^\d$', ['5', '0', 'a', '12', ''], "\d")

In [ ]:
# \w = alphanumériquetest_regex(r'^\w+$', ['hello', 'abc_123', 'hello world', '@!?'], "\w+")

In [ ]:
# \s = espacetest_regex(r'^\s+$', [' ', '   ', '\t', 'abc', ''], "\s+ (que des espaces)")

## 3. Quantificateurs <a id='3-quantificateurs'></a>| Symbole | Signifie | Min | Max ||---|---|---|---|| `?` | 0 ou 1 (optionnel) | 0 | 1 || `+` | 1 ou plus | 1 | ∞ || `*` | 0 ou plus | 0 | ∞ || `{n}` | exactement n fois | n | n || `{n,}` | au moins n fois | n | ∞ || `{n,m}` | entre n et m fois | n | m |⚠️ Un quantificateur **s'applique au caractère ou groupe juste avant**.

In [ ]:
# ? = 0 ou 1test_regex(r'^colou?r$', ['color', 'colour', 'colouur'], "colou?r")

In [ ]:
# + = 1 ou plustest_regex(r'^\d+$', ['', '5', '42', '12345', 'abc'], "\d+")

In [ ]:
# * = 0 ou plustest_regex(r'^\d*$', ['', '5', '42', '12345', 'abc'], "\d* (le vide passe !)")

In [ ]:
# {n} = exactement ntest_regex(r'^\d{4}$', ['2022', '202', '20222', '2022a'], "\d{4}")

In [ ]:
# {n,m} = entre n et mtest_regex(r'^\d{2,3}$', ['1', '12', '123', '1234'], "\d{2,3}")

### 🎓 Comparaison `+` vs `*` vs `?`Mémorise cette différence avec un seul tableau :| Données | `^\d+$` | `^\d*$` | `^\d?$` ||---|---|---|---|| `""` (vide) | ❌ | ✅ | ✅ || `"5"` | ✅ | ✅ | ✅ || `"42"` | ✅ | ✅ | ❌ || `"12345"` | ✅ | ✅ | ❌ |

In [ ]:
# Démonstration en directprint("=" * 50)test_regex(r'^\d+$', ['', '5', '42', '12345'], "\d+ (au moins 1)")test_regex(r'^\d*$', ['', '5', '42', '12345'], "\d* (zéro ou plus)")test_regex(r'^\d?$', ['', '5', '42', '12345'], "\d? (zéro ou un)")

## 4. Ancres de position <a id='4-ancres'></a>| Symbole | Signifie ||---|---|| `^` | début de chaîne || `$` | fin de chaîne || `\b` | frontière de mot |⚠️ **Sans ancres**, regex cherche **n'importe où** dans la chaîne. Pour valider une chaîne entière → toujours `^...$`.

In [ ]:
# Sans ancre : trouve \d+ n'importe oùimport reprint(re.search(r'\d+', 'abc123def'))   # match '123'print(re.search(r'\d+', 'no digits'))   # None

In [ ]:
# Avec ancres : la chaîne ENTIÈRE doit matchertest_regex(r'\d+', ['abc123def', '123', 'abc'], "\d+ sans ancre (trouve n'importe où)")test_regex(r'^\d+$', ['abc123def', '123', 'abc'], "^\d+$ (chaîne entière)")

## 5. Classes de caractères `[...]` <a id='5-classes'></a>Permettent de définir un **ensemble de caractères acceptés à une position**.| Pattern | Signifie ||---|---|| `[abc]` | un seul caractère parmi a, b, c || `[^abc]` | un caractère SAUF a, b, c || `[a-z]` | une minuscule || `[A-Z]` | une majuscule || `[0-9]` | un chiffre (équivalent à `\d`) || `[a-zA-Z0-9_]` | équivalent à `\w` |🧠 À l'intérieur des `[]`, la plupart des symboles spéciaux **perdent leur sens** : `[.+*]` = un point, un plus ou une étoile littéraux.

In [ ]:
# [abc] = un caractère parmi a, b, ctest_regex(r'^[aeiou]$', ['a', 'e', 'b', 'aa'], "Une voyelle")

In [ ]:
# [^abc] = négation : tout SAUF a, b, ctest_regex(r'^[^0-9]+$', ['hello', 'abc', 'abc123', '123'], "Pas de chiffre")

In [ ]:
# Plage [a-z], [A-Z], [0-9]test_regex(r'^[A-Z]\d+$', ['A3', 'A4', 'S10', 'a3', '3A'], "Majuscule + chiffres (codes SANDRE)")

In [ ]:
# [.,] = soit un point soit une virguletest_regex(r'^\d+[.,]\d+$', ['3.14', '3,14', '3 14', '3.14.15'], "Décimal FR ou EN")

## 6. Groupes et alternatives <a id='6-groupes'></a>### Parenthèses `(...)` — groupementPermettent d'appliquer un quantificateur à un **groupe** entier.### Alternative `|` — OU logiqueMatch l'un OU l'autre.

In [ ]:
# (ab)+ = répéter le groupe "ab"test_regex(r'^(ab)+$', ['ab', 'abab', 'ababab', 'abc', 'a'], "(ab)+")

In [ ]:
# Alternative |test_regex(r'^(chat|chien|hamster)$', ['chat', 'chien', 'hamster', 'oiseau'], "chat|chien|hamster")

In [ ]:
# Cas avancé : mois 1-12test_regex(r'^(1[0-2]|[1-9])$', ['1', '9', '10', '12', '0', '13', '99'], "Mois (1-12)")

## 7. Négation et lookahead <a id='7-negation'></a>| Pattern | Signifie ||---|---|| `[^abc]` | un caractère **sauf** a, b ou c (dans une classe) || `(?!...)` | **negative lookahead** : pas suivi de... || `(?=...)` | **positive lookahead** : suivi de... |⚠️ **Les lookaheads sont avancés.** En PySpark, préfère `~F.col(...).rlike(...)` pour inverser un match.

In [ ]:
# Negative lookahead : tout SAUF "12 chiffres exacts"test_regex(r'^(?!\d{12}$).+$', ['030201101000', '0435162S0001', '12345', 'ABC'], "Pas exactement 12 chiffres")

## 8. Raw string `r'...'` <a id='8-raw-string'></a>En Python, le `\` est un **caractère d'échappement** (ex: `\n` = retour ligne).Pour que la regex voie `\d` comme "**backslash + d**" et pas "essaie d'interpréter `\d`", on utilise `r'...'`.

In [ ]:
# DANGER : sans raw stringprint('\n')          # affiche un retour ligneprint('\d')          # affiche \d (par chance, mais c'est fragile)print(r'\d')         # raw string : safe ✅# 🧠 Règle d'or : TOUJOURS utiliser r'...' pour les regex en Python

## 9. Recettes prêtes à l'emploi <a id='9-recettes'></a>Tableau des regex courantes que tu peux **copier-coller** dans tes projets.

In [ ]:
# Validation de formats courantsrecettes = {    "Année (4 chiffres)":          r'^\d{4}$',    "Mois (1-12)":                 r'^(1[0-2]|[1-9])$',    "Date FR (jj/mm/aaaa)":        r'^\d{2}/\d{2}/\d{4}$',    "Heure (HH:MM)":               r'^([01]\d|2[0-3]):[0-5]\d$',    "Entier positif":              r'^\d+$',    "Entier signé":                r'^-?\d+$',    "Décimal FR/EN":               r'^-?\d+([.,]\d+)?$',    "Code postal FR":              r'^\d{5}$',    "Email simple":                r'^[\w.-]+@[\w.-]+\.\w+$',    "URL simple":                  r'^https?://\S+$',}for nom, pattern in recettes.items():    print(f"{nom:<35} {pattern}")

In [ ]:
# Test des recettes en actiontest_regex(recettes["Année (4 chiffres)"],     ['2022', '22', '2022a'])test_regex(recettes["Date FR (jj/mm/aaaa)"],   ['15/12/2022', '15-12-2022', '2022/12/15'])test_regex(recettes["Décimal FR/EN"],          ['1200', '-50', '1200.5', '1200,5', '1.2.3'])test_regex(recettes["Email simple"],           ['john@example.com', 'invalid', 'a@b.c'])

### 🎯 Recettes spéciales SANDRE (pour ton projet STEU)

In [ ]:
recettes_sandre = {    "Code SANDRE strict (12 chiffres)":  r'^\d{12}$',    "Code SANDRE flexible":               r'^\d+[A-Z]?\d*$',    "Point de prélèvement":               r'^[A-Z]\d+[a-z]?$',    "Département FR (avec DOM)":          r'^(\d{2}|97[1-6])$',    "Numérique avec espaces tolérés":     r'^\s*-?\d+([.,]\d+)?\s*$',}# Test sur exemples concretstest_regex(recettes_sandre["Code SANDRE flexible"],           ['030201101000', '0435162S0001', '123ABC', 'S0001'],           "Codes SANDRE")test_regex(recettes_sandre["Point de prélèvement"],           ['A3', 'A4', 'S10', 'M1', 'Q1b', 'a3', 'AA3'],           "Points SANDRE")test_regex(recettes_sandre["Département FR (avec DOM)"],           ['02', '75', '971', '974', '2A', '999'],           "Départements")

## 10. Spécial PySpark <a id='10-pyspark'></a>PySpark utilise `rlike` (= "regex like") pour le matching.```pythonfrom pyspark.sql import functions as F# Garder ce qui matchedf.filter(F.col("AnPrel").rlike(r'^\d{4}$'))# Garder ce qui NE matche PASdf.filter(~F.col("AnPrel").rlike(r'^\d{4}$'))# Combiner avec NULL safetydf.filter(F.col("AnPrel").rlike(r'^\d{4}$') & F.col("AnPrel").isNotNull())```### Substitution avec `regexp_replace````python# Virgule décimale → pointdf.withColumn("RsAna", F.regexp_replace(F.col("RsAna"), ',', '.'))# Retirer les guillemets en début/findf.withColumn("col", F.regexp_replace(F.col("col"), r'^"|"$', ''))```### Extraction avec `regexp_extract````python# Extraire la 1ère partie d'un groupe (groupe 1)df.withColumn("annee", F.regexp_extract(F.col("DtPrel"), r'/(\d{4})$', 1))```

## 11. Pièges classiques à éviter <a id='11-pieges'></a>| Piège | Pourquoi | Solution ||---|---|---|| Oublier `r''` | `\d` mal interprété | Toujours `r'...'` || Oublier `^` et `$` | Match partiel inattendu | Ancrer avec `^...$` || `\d` au lieu de `\d+` | Match d'**un seul** caractère | Mettre un quantificateur || `.` au lieu de `\.` | `.` = n'importe quoi | Échapper avec `\.` quand on veut un point || `*` au lieu de `+` | Accepte le vide | Choisir selon le besoin || `rlike` sur NULL | Renvoie NULL, pas False | Combiner avec `isNotNull()` |

In [ ]:
# Démo du piège : '.' au lieu de '\.'print("Sans échapper le point :")test_regex(r'^3.14$', ['3.14', '3a14', '3.145'], "3.14 (.)")print("Avec échappement \.:")test_regex(r'^3\.14$', ['3.14', '3a14', '3.145'], "3.14 (\.)")

## 12. Workflow d'écriture d'une regex <a id='12-workflow'></a>Quand tu dois écrire une nouvelle regex, suis ces 7 étapes :```1. EXEMPLE     → prends 3-5 valeurs valides + 3-5 invalides2. PATTERN     → décris le format en français ("des chiffres + un slash + des chiffres")3. TRADUCTION  → convertis en regex (\d+/\d+)4. ANCRES      → ajoute ^...$ pour valider toute la chaîne5. TEST        → utilise test_regex() ou regex101.com6. INTÉGRATION → copie dans ton code Python avec r'...'7. VALIDATION  → lance sur ton dataset réel, regarde les anomalies```### 🎓 Exercice — à toi de jouer !Écris une regex qui matche un **identifiant de produit** au format `PROD-12345` (`PROD-` suivi de 5 chiffres exactement).

In [ ]:
# 🎓 Solution proposée — modifie le pattern et testema_regex = r'^PROD-\d{5}$'valeurs_test = [    'PROD-12345',    # valide    'PROD-1234',     # 4 chiffres seulement    'PROD-123456',   # 6 chiffres    'PROD12345',     # pas de tiret    'prod-12345',    # minuscules    'PROD-abcde',    # lettres au lieu de chiffres]test_regex(ma_regex, valeurs_test, "Produit PROD-12345")

## 🌐 Ressources externes- 🔗 [regex101.com](https://regex101.com/) — testeur interactif (mode Python)- 🔗 [Python re HOWTO](https://docs.python.org/3/howto/regex.html) — tutoriel officiel- 🔗 [Python re docs](https://docs.python.org/3/library/re.html) — référence complète- 🔗 [Spark functions](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.sql.functions.html) — `regexp_replace`, `regexp_extract`, `rlike`---## 🧠 Récap mnémotechnique```.    n'importe quel caractère\d   un chiffre        \D   pas un chiffre\w   alphanumérique    \W   pas alphanumérique\s   espace            \S   pas un espace^    début             $    fin[]   classe            [^]  classe négative?    0 ou 1            *    0 ou plus           +    1 ou plus{n}  exactement n      {n,m} entre n et m()   groupe            |    OU\    échappement       r''  raw string Python```🎯 **Bookmark ce notebook**. Reviens-y dès que tu butes sur une regex. Bon code ! 💪